In [411]:
import pandas as pd
from pathlib import Path
import os
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [412]:
DATA_DIR = Path(os.getcwd()).parent / "data"

In [413]:
anime_df = pd.read_csv(DATA_DIR / "anime.csv")
anime_df.head()

,MAL_ID,Name,Score,Genres,English name,Japanese name,Type,Episodes,Aired,Premiered,...,Score-10,Score-9,Score-8,Score-7,Score-6,Score-5,Score-4,Score-3,Score-2,Score-1
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",Cowboy Bebop,カウボーイビバップ,TV,26,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,...,229170.0,182126.0,131625.0,62330.0,20688.0,8904.0,3184.0,1357.0,741.0,1580.0
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space",Cowboy Bebop:The Movie,カウボーイビバップ 天国の扉,Movie,1,"Sep 1, 2001",Unknown,...,30043.0,49201.0,49505.0,22632.0,5805.0,1877.0,577.0,221.0,109.0,379.0
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",Trigun,トライガン,TV,26,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,...,50229.0,75651.0,86142.0,49432.0,15376.0,5838.0,1965.0,664.0,316.0,533.0
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",Witch Hunter Robin,Witch Hunter ROBIN (ウイッチハンターロビン),TV,26,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,...,2182.0,4806.0,10128.0,11618.0,5709.0,2920.0,1083.0,353.0,164.0,131.0
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",Beet the Vandel Buster,冒険王ビィト,TV,52,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,...,312.0,529.0,1242.0,1713.0,1068.0,634.0,265.0,83.0,50.0,27.0


In [414]:
len(anime_df)

17562

In [415]:
anime_synopsis = pd.read_csv(DATA_DIR / "anime_with_synopsis.csv")
anime_synopsis.head()

,MAL_ID,Name,Score,Genres,synopsis
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space","In the year 2071, humanity has colonized sever..."
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space","other day, another bounty—such is the life of ..."
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen","Vash the Stampede is the man with a $$60,000,0..."
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",ches are individuals with special powers like ...
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",It is the dark century and the people are suff...


In [416]:
len(anime_synopsis)

16214

In [417]:
def replace_unknown_with_nan(df):
    """
    Replace all forms of 'unknown' with NaN in entire dataframe.
    Case-insensitive.
    """
    return df.replace(r'(?i)^unknown$', np.nan, regex=True)

In [418]:
anime_df = replace_unknown_with_nan(anime_df)
anime_synopsis = replace_unknown_with_nan(anime_synopsis)

In [419]:
anime_df.head()

,MAL_ID,Name,Score,Genres,English name,Japanese name,Type,Episodes,Aired,Premiered,...,Score-10,Score-9,Score-8,Score-7,Score-6,Score-5,Score-4,Score-3,Score-2,Score-1
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",Cowboy Bebop,カウボーイビバップ,TV,26,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,...,229170.0,182126.0,131625.0,62330.0,20688.0,8904.0,3184.0,1357.0,741.0,1580.0
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space",Cowboy Bebop:The Movie,カウボーイビバップ 天国の扉,Movie,1,"Sep 1, 2001",NaN,...,30043.0,49201.0,49505.0,22632.0,5805.0,1877.0,577.0,221.0,109.0,379.0
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",Trigun,トライガン,TV,26,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,...,50229.0,75651.0,86142.0,49432.0,15376.0,5838.0,1965.0,664.0,316.0,533.0
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",Witch Hunter Robin,Witch Hunter ROBIN (ウイッチハンターロビン),TV,26,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,...,2182.0,4806.0,10128.0,11618.0,5709.0,2920.0,1083.0,353.0,164.0,131.0
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",Beet the Vandel Buster,冒険王ビィト,TV,52,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,...,312.0,529.0,1242.0,1713.0,1068.0,634.0,265.0,83.0,50.0,27.0


In [420]:
anime_df.isnull().sum()

MAL_ID               0
Name                 0
Score             5141
Genres              63
English name     10565
Japanese name       48
Type                37
Episodes           516
Aired              309
Premiered        12817
Producers         7794
Licensors        13616
Studios           7079
Source            3567
Duration           555
Rating             688
Ranked            1762
Popularity           0
Members              0
Favorites            0
Watching             0
Completed            0
On-Hold              0
Dropped              0
Plan to Watch        0
Score-10           437
Score-9           3167
Score-8           1371
Score-7            503
Score-6            511
Score-5            584
Score-4            977
Score-3           1307
Score-2           1597
Score-1            459
dtype: int64

In [421]:
anime_synopsis.isnull().sum()

MAL_ID         0
Name           0
Score       5123
Genres        63
synopsis       8
dtype: int64

Cleaning Anime df

In [422]:
anime_df.columns

Index(['MAL_ID', 'Name', 'Score', 'Genres', 'English name', 'Japanese name',
       'Type', 'Episodes', 'Aired', 'Premiered', 'Producers', 'Licensors',
       'Studios', 'Source', 'Duration', 'Rating', 'Ranked', 'Popularity',
       'Members', 'Favorites', 'Watching', 'Completed', 'On-Hold', 'Dropped',
       'Plan to Watch', 'Score-10', 'Score-9', 'Score-8', 'Score-7', 'Score-6',
       'Score-5', 'Score-4', 'Score-3', 'Score-2', 'Score-1'],
      dtype='object')

In [423]:
cols_to_drop = [
    'Score', 'Japanese name', 'Type', 'Licensors', 'Ranked', 'Popularity', 'Members', 'Favorites', 
    'Watching', 'Completed', 'On-Hold', 'Dropped', 'Plan to Watch', 
    'Score-10', 'Score-9', 'Score-8', 'Score-7', 'Score-6', 
    'Score-5', 'Score-4', 'Score-3', 'Score-2', 'Score-1'
]

anime_df = anime_df.drop(columns=cols_to_drop, errors='ignore')
anime_df.head()

,MAL_ID,Name,Genres,English name,Episodes,Aired,Premiered,Producers,Studios,Source,Duration,Rating
0,1,Cowboy Bebop,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",Cowboy Bebop,26,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,Bandai Visual,Sunrise,Original,24 min. per ep.,R - 17+ (violence & profanity)
1,5,Cowboy Bebop: Tengoku no Tobira,"Action, Drama, Mystery, Sci-Fi, Space",Cowboy Bebop:The Movie,1,"Sep 1, 2001",NaN,"Sunrise, Bandai Visual",Bones,Original,1 hr. 55 min.,R - 17+ (violence & profanity)
2,6,Trigun,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",Trigun,26,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,Victor Entertainment,Madhouse,Manga,24 min. per ep.,PG-13 - Teens 13 or older
3,7,Witch Hunter Robin,"Action, Mystery, Police, Supernatural, Drama, ...",Witch Hunter Robin,26,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,"TV Tokyo, Bandai Visual, Dentsu, Victor Entert...",Sunrise,Original,25 min. per ep.,PG-13 - Teens 13 or older
4,8,Bouken Ou Beet,"Adventure, Fantasy, Shounen, Supernatural",Beet the Vandel Buster,52,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,"TV Tokyo, Dentsu",Toei Animation,Manga,23 min. per ep.,PG - Children


In [424]:
anime_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17562 entries, 0 to 17561
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   MAL_ID        17562 non-null  int64 
 1   Name          17562 non-null  object
 2   Genres        17499 non-null  object
 3   English name  6997 non-null   object
 4   Episodes      17046 non-null  object
 5   Aired         17253 non-null  object
 6   Premiered     4745 non-null   object
 7   Producers     9768 non-null   object
 8   Studios       10483 non-null  object
 9   Source        13995 non-null  object
 10  Duration      17007 non-null  object
 11  Rating        16874 non-null  object
dtypes: int64(1), object(11)
memory usage: 1.6+ MB


In [425]:
anime_df['Episodes'] = pd.to_numeric(anime_df['Episodes'],errors='coerce').astype('Int64')

def extract_duration_minutes(df, col='Duration'):
    """
    Extract numeric minutes from duration column.
    Example:
    '24 min. per ep.' -> 24
    """
    
    df[col] = (df[col].str.extract(r'(\d+)')[0].astype('Int64'))
    return df

anime_df = extract_duration_minutes(anime_df)

anime_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17562 entries, 0 to 17561
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   MAL_ID        17562 non-null  int64 
 1   Name          17562 non-null  object
 2   Genres        17499 non-null  object
 3   English name  6997 non-null   object
 4   Episodes      17046 non-null  Int64 
 5   Aired         17253 non-null  object
 6   Premiered     4745 non-null   object
 7   Producers     9768 non-null   object
 8   Studios       10483 non-null  object
 9   Source        13995 non-null  object
 10  Duration      17007 non-null  Int64 
 11  Rating        16874 non-null  object
dtypes: Int64(2), int64(1), object(9)
memory usage: 1.6+ MB


Cleaning Anime synopsis df

In [426]:
anime_synopsis.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16214 entries, 0 to 16213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   MAL_ID    16214 non-null  int64 
 1   Name      16214 non-null  object
 2   Score     11091 non-null  object
 3   Genres    16151 non-null  object
 4   synopsis  16206 non-null  object
dtypes: int64(1), object(4)
memory usage: 633.5+ KB


In [427]:
cols_to_drop = ['Score']

anime_synopsis = anime_synopsis.drop(columns=cols_to_drop, errors='ignore')
anime_synopsis.head()

,MAL_ID,Name,Genres,synopsis
0,1,Cowboy Bebop,"Action, Adventure, Comedy, Drama, Sci-Fi, Space","In the year 2071, humanity has colonized sever..."
1,5,Cowboy Bebop: Tengoku no Tobira,"Action, Drama, Mystery, Sci-Fi, Space","other day, another bounty—such is the life of ..."
2,6,Trigun,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen","Vash the Stampede is the man with a $$60,000,0..."
3,7,Witch Hunter Robin,"Action, Mystery, Police, Supernatural, Drama, ...",ches are individuals with special powers like ...
4,8,Bouken Ou Beet,"Adventure, Fantasy, Shounen, Supernatural",It is the dark century and the people are suff...


Merging 

In [428]:
df = pd.merge(anime_df, anime_synopsis, on='MAL_ID', how='left')
df.head()

,MAL_ID,Name_x,Genres_x,English name,Episodes,Aired,Premiered,Producers,Studios,Source,Duration,Rating,Name_y,Genres_y,synopsis
0,1,Cowboy Bebop,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",Cowboy Bebop,26,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,Bandai Visual,Sunrise,Original,24,R - 17+ (violence & profanity),Cowboy Bebop,"Action, Adventure, Comedy, Drama, Sci-Fi, Space","In the year 2071, humanity has colonized sever..."
1,5,Cowboy Bebop: Tengoku no Tobira,"Action, Drama, Mystery, Sci-Fi, Space",Cowboy Bebop:The Movie,1,"Sep 1, 2001",NaN,"Sunrise, Bandai Visual",Bones,Original,1,R - 17+ (violence & profanity),Cowboy Bebop: Tengoku no Tobira,"Action, Drama, Mystery, Sci-Fi, Space","other day, another bounty—such is the life of ..."
2,6,Trigun,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",Trigun,26,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,Victor Entertainment,Madhouse,Manga,24,PG-13 - Teens 13 or older,Trigun,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen","Vash the Stampede is the man with a $$60,000,0..."
3,7,Witch Hunter Robin,"Action, Mystery, Police, Supernatural, Drama, ...",Witch Hunter Robin,26,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,"TV Tokyo, Bandai Visual, Dentsu, Victor Entert...",Sunrise,Original,25,PG-13 - Teens 13 or older,Witch Hunter Robin,"Action, Mystery, Police, Supernatural, Drama, ...",ches are individuals with special powers like ...
4,8,Bouken Ou Beet,"Adventure, Fantasy, Shounen, Supernatural",Beet the Vandel Buster,52,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,"TV Tokyo, Dentsu",Toei Animation,Manga,23,PG - Children,Bouken Ou Beet,"Adventure, Fantasy, Shounen, Supernatural",It is the dark century and the people are suff...


preprocessing merged df

In [429]:
def create_title_column(df):
    """
    Create 'title' column with priority:
    Name_x -> Name_y -> English name -> NaN
    """
    df['Title'] = (df['Name_x'].combine_first(df['Name_y']).combine_first(df['English name']))
    return df

def merge_genres(df):
    """
    Create 'genres' column by taking union of
    Genres_x and Genres_y (comma-separated values).
    """
    def union_genres(row):
        g1 = row['Genres_x']
        g2 = row['Genres_y']
        set1 = set(map(str.strip, g1.split(','))) if pd.notna(g1) else set()
        set2 = set(map(str.strip, g2.split(','))) if pd.notna(g2) else set()
        merged = sorted(set1 | set2)
        return ', '.join(merged) if merged else np.nan
    df['Genres'] = df.apply(union_genres, axis=1)
    return df

def cleanup_columns(df):
    """
    Drop old name and genre columns.
    """
    cols_to_drop = ['Name_x', 'Name_y', 'Genres_x', 'Genres_y', 'English name']
    return df.drop(columns=cols_to_drop)

def process_anime_df(df):
    """
    Complete pipeline.
    """
    df = create_title_column(df)
    df = merge_genres(df)
    df = cleanup_columns(df)
    return df

In [430]:
df = process_anime_df(df)
df.head()

,MAL_ID,Episodes,Aired,Premiered,Producers,Studios,Source,Duration,Rating,synopsis,Title,Genres
0,1,26,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,Bandai Visual,Sunrise,Original,24,R - 17+ (violence & profanity),"In the year 2071, humanity has colonized sever...",Cowboy Bebop,"Action, Adventure, Comedy, Drama, Sci-Fi, Space"
1,5,1,"Sep 1, 2001",NaN,"Sunrise, Bandai Visual",Bones,Original,1,R - 17+ (violence & profanity),"other day, another bounty—such is the life of ...",Cowboy Bebop: Tengoku no Tobira,"Action, Drama, Mystery, Sci-Fi, Space"
2,6,26,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,Victor Entertainment,Madhouse,Manga,24,PG-13 - Teens 13 or older,"Vash the Stampede is the man with a $$60,000,0...",Trigun,"Action, Adventure, Comedy, Drama, Sci-Fi, Shounen"
3,7,26,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,"TV Tokyo, Bandai Visual, Dentsu, Victor Entert...",Sunrise,Original,25,PG-13 - Teens 13 or older,ches are individuals with special powers like ...,Witch Hunter Robin,"Action, Drama, Magic, Mystery, Police, Superna..."
4,8,52,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,"TV Tokyo, Dentsu",Toei Animation,Manga,23,PG - Children,It is the dark century and the people are suff...,Bouken Ou Beet,"Adventure, Fantasy, Shounen, Supernatural"


In [431]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17562 entries, 0 to 17561
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   MAL_ID     17562 non-null  int64 
 1   Episodes   17046 non-null  Int64 
 2   Aired      17253 non-null  object
 3   Premiered  4745 non-null   object
 4   Producers  9768 non-null   object
 5   Studios    10483 non-null  object
 6   Source     13995 non-null  object
 7   Duration   17007 non-null  Int64 
 8   Rating     16874 non-null  object
 9   synopsis   16206 non-null  object
 10  Title      17562 non-null  object
 11  Genres     17499 non-null  object
dtypes: Int64(2), int64(1), object(9)
memory usage: 1.6+ MB


cleaning merged df

In [432]:
df['Source'].value_counts()

Source
Original         5215
Manga            3825
Visual novel      993
Game              880
Light novel       768
Other             597
Novel             510
Music             317
4-koma manga      288
Web manga         252
Picture book      147
Book              112
Card game          64
Digital manga      15
Radio              12
Name: count, dtype: int64

In [433]:
df['Source'] = df['Source'].fillna('Original')

In [434]:
df['Episodes'] = df.groupby('Source')['Episodes'].transform(lambda x: x.fillna(x.median()))
df['Episodes'] = df['Episodes'].fillna(df['Episodes'].median())


df['Ep_bin'] = pd.cut(df['Episodes'],
    bins=[0, 1, 13, 26, 52, 100, float('inf')],
    labels=['movie_ova', 'short', 'one_cour', 'two_cour', 'long', 'ongoing']
)

df.drop(columns=['Episodes'], inplace=True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17562 entries, 0 to 17561
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   MAL_ID     17562 non-null  int64   
 1   Aired      17253 non-null  object  
 2   Premiered  4745 non-null   object  
 3   Producers  9768 non-null   object  
 4   Studios    10483 non-null  object  
 5   Source     17562 non-null  object  
 6   Duration   17007 non-null  Int64   
 7   Rating     16874 non-null  object  
 8   synopsis   16206 non-null  object  
 9   Title      17562 non-null  object  
 10  Genres     17499 non-null  object  
 11  Ep_bin     17562 non-null  category
dtypes: Int64(1), category(1), int64(1), object(9)
memory usage: 1.5+ MB


In [435]:
df.dropna(subset=['synopsis'], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16206 entries, 0 to 17561
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   MAL_ID     16206 non-null  int64   
 1   Aired      15900 non-null  object  
 2   Premiered  4740 non-null   object  
 3   Producers  8753 non-null   object  
 4   Studios    10045 non-null  object  
 5   Source     16206 non-null  object  
 6   Duration   15659 non-null  Int64   
 7   Rating     15523 non-null  object  
 8   synopsis   16206 non-null  object  
 9   Title      16206 non-null  object  
 10  Genres     16143 non-null  object  
 11  Ep_bin     16206 non-null  category
dtypes: Int64(1), category(1), int64(1), object(9)
memory usage: 1.5+ MB


In [436]:
df['Premiered'] = df['Premiered'].fillna('Unknown')
df['Producers'] = df['Producers'].fillna('Unknown')
df['Studios'] = df['Studios'].fillna('Unknown')
df['Rating'] = df['Rating'].fillna(df['Rating'].mode()[0])
df['Genres'] = df['Genres'].fillna('Unknown')

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16206 entries, 0 to 17561
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   MAL_ID     16206 non-null  int64   
 1   Aired      15900 non-null  object  
 2   Premiered  16206 non-null  object  
 3   Producers  16206 non-null  object  
 4   Studios    16206 non-null  object  
 5   Source     16206 non-null  object  
 6   Duration   15659 non-null  Int64   
 7   Rating     16206 non-null  object  
 8   synopsis   16206 non-null  object  
 9   Title      16206 non-null  object  
 10  Genres     16206 non-null  object  
 11  Ep_bin     16206 non-null  category
dtypes: Int64(1), category(1), int64(1), object(9)
memory usage: 1.5+ MB


In [437]:
df['Duration'] = df.groupby('Ep_bin', observed=False)['Duration'].transform(lambda x: x.fillna(x.median()))
df['Duration'] = df['Duration'].fillna(df['Duration'].median())

In [438]:
df.groupby('Ep_bin', observed=False)['Duration'].median()

Ep_bin
movie_ova     5.0
short        23.0
one_cour     23.0
two_cour     24.0
long         24.0
ongoing      22.0
Name: Duration, dtype: Float64

In [439]:
df[df['Ep_bin'] == 'movie_ova'].groupby('Source')['Duration'].median()

Source
4-koma manga      8.5
Book             11.5
Card game        24.0
Digital manga    28.0
Game              5.0
Light novel      23.0
Manga            14.0
Music             4.0
Novel             1.0
Original          5.0
Other             5.5
Picture book     11.5
Radio            15.0
Visual novel     23.0
Web manga         4.0
Name: Duration, dtype: Float64

In [440]:
df.head()

,MAL_ID,Aired,Premiered,Producers,Studios,Source,Duration,Rating,synopsis,Title,Genres,Ep_bin
0,1,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,Bandai Visual,Sunrise,Original,24,R - 17+ (violence & profanity),"In the year 2071, humanity has colonized sever...",Cowboy Bebop,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",one_cour
1,5,"Sep 1, 2001",Unknown,"Sunrise, Bandai Visual",Bones,Original,1,R - 17+ (violence & profanity),"other day, another bounty—such is the life of ...",Cowboy Bebop: Tengoku no Tobira,"Action, Drama, Mystery, Sci-Fi, Space",movie_ova
2,6,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,Victor Entertainment,Madhouse,Manga,24,PG-13 - Teens 13 or older,"Vash the Stampede is the man with a $$60,000,0...",Trigun,"Action, Adventure, Comedy, Drama, Sci-Fi, Shounen",one_cour
3,7,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,"TV Tokyo, Bandai Visual, Dentsu, Victor Entert...",Sunrise,Original,25,PG-13 - Teens 13 or older,ches are individuals with special powers like ...,Witch Hunter Robin,"Action, Drama, Magic, Mystery, Police, Superna...",one_cour
4,8,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,"TV Tokyo, Dentsu",Toei Animation,Manga,23,PG - Children,It is the dark century and the people are suff...,Bouken Ou Beet,"Adventure, Fantasy, Shounen, Supernatural",two_cour


In [441]:
df['Dur_bin'] = pd.cut(df['Duration'],bins=[0, 10, 30, 60, float('inf')],labels=['short_form', 'standard', 'ova_length', 'movie_length'])
df.drop(columns=['Duration'], inplace=True)
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 16206 entries, 0 to 17561
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   MAL_ID     16206 non-null  int64   
 1   Aired      15900 non-null  object  
 2   Premiered  16206 non-null  object  
 3   Producers  16206 non-null  object  
 4   Studios    16206 non-null  object  
 5   Source     16206 non-null  object  
 6   Rating     16206 non-null  object  
 7   synopsis   16206 non-null  object  
 8   Title      16206 non-null  object  
 9   Genres     16206 non-null  object  
 10  Ep_bin     16206 non-null  category
 11  Dur_bin    16206 non-null  category
dtypes: category(2), int64(1), object(9)
memory usage: 1.4+ MB


In [442]:
df['start_date'] = pd.to_datetime(df['Aired'].str.extract(r'^([A-Za-z]+ \d+, \d{4})')[0], errors='coerce')

df['year_from_premiered'] = df['Premiered'].str.extract(r'(\d{4})').astype(float)
df['start_year'] = df['start_date'].dt.year
df['start_year'] = df['start_year'].fillna(df['year_from_premiered'])
df['start_year'] = df['start_year'].fillna(df['start_year'].median())

df['Era'] = pd.cut(df['start_year'],bins=[0, 1989, 1999, 2009, 2019, float('inf')],labels=['classic', '90s', '2000s', '2010s', '2020s'])

df.drop(columns=['Aired', 'start_date', 'year_from_premiered', 'start_year', 'Premiered'], inplace=True)
df.head()

,MAL_ID,Producers,Studios,Source,Rating,synopsis,Title,Genres,Ep_bin,Dur_bin,Era
0,1,Bandai Visual,Sunrise,Original,R - 17+ (violence & profanity),"In the year 2071, humanity has colonized sever...",Cowboy Bebop,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",one_cour,standard,90s
1,5,"Sunrise, Bandai Visual",Bones,Original,R - 17+ (violence & profanity),"other day, another bounty—such is the life of ...",Cowboy Bebop: Tengoku no Tobira,"Action, Drama, Mystery, Sci-Fi, Space",movie_ova,short_form,2000s
2,6,Victor Entertainment,Madhouse,Manga,PG-13 - Teens 13 or older,"Vash the Stampede is the man with a $$60,000,0...",Trigun,"Action, Adventure, Comedy, Drama, Sci-Fi, Shounen",one_cour,standard,90s
3,7,"TV Tokyo, Bandai Visual, Dentsu, Victor Entert...",Sunrise,Original,PG-13 - Teens 13 or older,ches are individuals with special powers like ...,Witch Hunter Robin,"Action, Drama, Magic, Mystery, Police, Superna...",one_cour,standard,2000s
4,8,"TV Tokyo, Dentsu",Toei Animation,Manga,PG - Children,It is the dark century and the people are suff...,Bouken Ou Beet,"Adventure, Fantasy, Shounen, Supernatural",two_cour,standard,2000s


In [443]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16206 entries, 0 to 17561
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   MAL_ID     16206 non-null  int64   
 1   Producers  16206 non-null  object  
 2   Studios    16206 non-null  object  
 3   Source     16206 non-null  object  
 4   Rating     16206 non-null  object  
 5   synopsis   16206 non-null  object  
 6   Title      16206 non-null  object  
 7   Genres     16206 non-null  object  
 8   Ep_bin     16206 non-null  category
 9   Dur_bin    16206 non-null  category
 10  Era        16206 non-null  category
dtypes: category(3), int64(1), object(7)
memory usage: 1.2+ MB


Content Based Recommender

In [444]:
def build_soup(row):
    """
    Build a "soup" of all relevant features for a given anime.
    """
    # Clean genres and producers (comma-separated to space-separated)
    genres    = str(row['Genres']).replace(',', ' ').replace('-', '')
    producers = str(row['Producers']).replace(',', ' ')
    studios   = str(row['Studios']).replace(',', ' ')
    source    = str(row['Source']).replace(' ', '_')
    rating    = str(row['Rating']).replace(' ', '_').replace('-', '')
    ep_bin    = str(row['Ep_bin'])
    dur_bin   = str(row['Dur_bin'])
    era       = str(row['Era'])
    synopsis  = str(row['synopsis'])
    title = str(row['Title']).replace(' ', '_').replace('-', '')

    # Repeat genres 2x to give them more weight than era/dur_bin
    return f"{genres} {genres} {studios} {producers} {source} {rating} {ep_bin} {dur_bin} {era} {synopsis} {title}"

df['soup'] = df.apply(build_soup, axis=1)

In [445]:
tfidf = TfidfVectorizer(stop_words='english', max_features=15000)
tfidf_matrix = tfidf.fit_transform(df['soup'])

print(tfidf_matrix.shape) 

(16206, 15000)


In [446]:
df = df.reset_index(drop=True)
title_to_idx = pd.Series(df.index, index=df['Title'].str.lower())

def recommend(title, n=10):
    """
    Recommend similar anime based on content."""
    title = title.lower()
    
    if title not in title_to_idx:
        print(f"'{title}' not found.")
        return None
    
    idx = title_to_idx[title]
    
    # Compute similarity only for this one anime vs all others
    query_vec = tfidf_matrix[idx]
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Get top N (exclude itself at index 0)
    top_indices = sim_scores.argsort()[::-1][1:n+1]
    
    results = df.iloc[top_indices][['Title', 'Genres', 'Studios', 'Era', 'Ep_bin']].copy()
    results['similarity'] = sim_scores[top_indices].round(3)
    
    return results.reset_index(drop=True)

In [447]:
recommend("Blue Gender", n=5)

,Title,Genres,Studios,Era,Ep_bin,similarity
0,Blue Gender: The Warrior,"Adventure, Drama, Horror, Mecha, Military, Rom...",Unknown,2000s,movie_ova,0.402
1,Uchuu Senkan Yamato (Movie),"Drama, Military, Sci-Fi, Space",Unknown,classic,movie_ova,0.273
2,Turn A Gundam,"Action, Adventure, Drama, Mecha, Military, Rom...","Sunrise, Nakamura Production",90s,two_cour,0.254
3,Tomica Kizuna Gattai: Earth Granner,"Action, Adventure, Mecha, Sci-Fi",OLM,2020s,movie_ova,0.234
4,Space Bug,"Adventure, Space","Studio W.Baba, P.I.C.S.",2010s,one_cour,0.229
